<a href="https://colab.research.google.com/github/nitijain18/style-buddy/blob/main/model3_sb_color.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opendatasets


In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small?utm_source=chatgpt.com")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: nitijain13
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small


100%|██████████| 565M/565M [00:10<00:00, 57.5MB/s]


In [ ]:
import pandas as pd
data = pd.read_csv("/content/fashion-product-images-small/styles.csv", on_bad_lines="skip")

In [ ]:
data.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [ ]:
data.columns

Index(['id', 'gender', 'masterCategory', 'subCategory', 'articleType',
       'baseColour', 'season', 'year', 'usage', 'productDisplayName'],
      dtype='object')

In [ ]:
data["baseColour"].value_counts()

,count
baseColour,
Black,9728
White,5538
Blue,4918
Brown,3494
Grey,2741
Red,2455
Green,2115
Pink,1860
Navy Blue,1789


In [ ]:
data["baseColour"] = data["baseColour"].replace ({
    "Taupe" : "Brown",
    "Lime Green": "Green",
    "Fluorescent Green": "Green",
    "Charcoal" : "Grey",
    "Steel" : "Grey",
    "Coffee Brown" : "Brown",
    "Mushroom Brown" : "Brown",
    "Rose" : "Pink",
    "Grey Melange" : "Grey",
    "Navy Blue" : "Blue",
    "Mauve" : "Purple",
    "Lavender" : "Purple",
    "Skin" : "Nude",
    "Tan" : "Nude",
    "Beige" : "Nude",
    "Khaki" : "Nude",
    "Gold" : "Metallic",
    "Copper" : "Metallic",
    "Bronze" : "Metallic",
    "Silver" : "Metallic",
    "Rust" : "Brown",
    "Turquoise Blue" : "Blue",
    "Teal" : "Blue",
    "Sea Green" : "Green",
    "Burgundy" : "Maroon",
    "Magenta" : "Pink",
    "Off White" : "Cream",
    "Mustard" : "Yellow",
    "Peach" : "Nude",
    "Olive" : "Green",
    "Cream" : "Nude",
    "metallic" : "Multi"

})

In [ ]:
data["baseColour"].value_counts()

,count
baseColour,
Black,9728
Blue,6896
White,5538
Brown,3618
Grey,3430
Green,2558
Red,2455
Pink,2017
Metallic,1942


In [ ]:
import os

image_dir = "/content/fashion-product-images-small/images"

data["image_path"] = data["id"].astype(str).apply(
    lambda x: os.path.join(image_dir, x + ".jpg")
)

data = data[data["image_path"].apply(os.path.exists)]

print(len(data))

44419


In [ ]:
data.isnull().sum()

,0
id,0
gender,0
masterCategory,0
subCategory,0
articleType,0
baseColour,15
season,21
year,1
usage,317
productDisplayName,7


In [ ]:
data = data.dropna(subset=["baseColour"])

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    stratify=data["baseColour"],
    random_state=42
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(rescale=1./255)

train_ds = train_gen.flow_from_dataframe(
    train_df,
    x_col="image_path",
    y_col="baseColour",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_ds = test_gen.flow_from_dataframe(
    test_df,
    x_col="image_path",
    y_col="baseColour",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 35523 validated image filenames belonging to 16 classes.
Found 8881 validated image filenames belonging to 16 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(16, activation="softmax")
])


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

Epoch 1/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 508s 436ms/step - accuracy: 0.3430 - loss: 2.0894 - val_accuracy: 0.4065 - val_loss: 1.8281
Epoch 2/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 459s 413ms/step - accuracy: 0.4186 - loss: 1.8184 - val_accuracy: 0.4507 - val_loss: 1.6972
Epoch 3/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 460s 414ms/step - accuracy: 0.4493 - loss: 1.7028 - val_accuracy: 0.4381 - val_loss: 1.7330
Epoch 4/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 463s 417ms/step - accuracy: 0.4744 - loss: 1.6264 - val_accuracy: 0.4511 - val_loss: 1.7168
Epoch 5/5
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 465s 419ms/step - accuracy: 0.4906 - loss: 1.5781 - val_accuracy: 0.5049 - val_loss: 1.5164


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)

278/278 ━━━━━━━━━━━━━━━━━━━━ 24s 86ms/step - accuracy: 0.5049 - loss: 1.5164
Test Accuracy: 0.5048980712890625


In [ ]:
import numpy as np
from sklearn.metrics import classification_report

pred = model.predict(test_ds)
y_pred = np.argmax(pred, axis=1)
y_true = test_ds.classes

print(classification_report(
    y_true,
    y_pred,
    target_names=list(test_ds.class_indices.keys()),
    zero_division=0
))

278/278 ━━━━━━━━━━━━━━━━━━━━ 32s 102ms/step
              precision    recall  f1-score   support

       Black       0.54      0.84      0.66      1945
        Blue       0.58      0.55      0.57      1379
       Brown       0.50      0.19      0.27       724
       Cream       0.00      0.00      0.00        36
       Green       0.71      0.29      0.41       512
        Grey       0.35      0.25      0.29       686
      Maroon       0.55      0.05      0.09       125
    Metallic       0.46      0.10      0.17       388
       Multi       0.50      0.01      0.02        79
        Nude       0.27      0.06      0.10       358
      Orange       0.63      0.18      0.28       106
        Pink       0.39      0.58      0.47       403
      Purple       0.39      0.27      0.32       366
         Red       0.65      0.45      0.53       491
       White       0.44      0.82      0.57      1108
      Yellow       0.68      0.50      0.57       175

    accuracy                        

In [ ]:
model.save("style_buddy_category_classifier-model3-color.keras")